# LSTM-arithmetic

## Dataset
- [Arithmetic dataset](https://drive.google.com/file/d/1cMuL3hF9jefka9RyF4gEBIGGeFGZYHE-/view?usp=sharing)

In [ ]:
# ! pip install seaborn
# ! pip install opencc
# ! pip install -U scikit-learn

import numpy as np
import pandas as pd
import torch
import torch.nn
import torch.nn.utils.rnn
import torch.utils.data
import matplotlib.pyplot as plt
import seaborn as sns
#import opencc
import os
from sklearn.model_selection import train_test_split

data_path = '/content/drive/MyDrive/nlp-hw2'

In [ ]:
df_train = pd.read_csv(os.path.join(data_path, 'arithmetic_train.csv'))
df_eval = pd.read_csv(os.path.join(data_path, 'arithmetic_eval.csv'))
df_train.head()

,src,tgt
0,(40*0)-18=,-18
1,6+4+31=,41
2,4*11+13=,57
3,44*39*26=,44616
4,(39*42)+27=,1665


In [ ]:
# transform the input data to string
df_train['tgt'] = df_train['tgt'].apply(lambda x: str(x))
df_train['src'] = df_train['src'].add(df_train['tgt'])
df_train['len'] = df_train['src'].apply(lambda x: len(x))

df_eval['tgt'] = df_eval['tgt'].apply(lambda x: str(x))

# Build Dictionary
 - The model cannot perform calculations directly with plain text.
 - Convert all text (numbers/symbols) into numerical representations.
 - Special tokens
    - '&lt;pad&gt;'
        - Each sentence within a batch may have different lengths.
        - The length is padded with '&lt;pad&gt;' to match the longest sentence in the batch.
    - '&lt;eos&gt;'
        - Specifies the end of the generated sequence.
        - Without '&lt;eos&gt;', the model will not know when to stop generating.

In [ ]:
char_to_id = {}
id_to_char = {}

# write your code here
# Build a dictionary and give every token in the train dataset an id
# The dictionary should contain <eos> and <pad>
# char_to_id is to conver charactors to ids, while id_to_char is the opposite

special_tokens = ['<pad>', '<eos>']
arithmetic_chars = ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '+', '-', '*', '(', ')', '=']
vocab = special_tokens + arithmetic_chars

char_to_id = {ch: idx for idx, ch in enumerate(vocab)}
id_to_char = {idx: ch for idx, ch in enumerate(vocab)}

vocab_size = len(char_to_id)
print('Vocab size{}'.format(vocab_size))

Vocab size18


# Data Preprocessing
 - The data is processed into the format required for the model's input and output. (End with \<eos\> token)


In [ ]:
# Write your code here
pad_id = char_to_id['<pad>']
eos_id = char_to_id['<eos>']

srcs = df_train['src'].values
tgts = df_train['tgt'].values

char_id_list = []
label_id_list = []

for s,t in zip(srcs,tgts):
    c_ids = [char_to_id[c] for c in s]
    prefix_len = len(s) - len(t)-1
    l_ids = [pad_id] * prefix_len + [char_to_id[c] for c in t] + [eos_id]
    char_id_list.append(c_ids)
    label_id_list.append(l_ids)

df_train['char_id_list'] = char_id_list
df_train['label_id_list'] = label_id_list

df_train.head()

,src,tgt,len,char_id_list,label_id_list
0,(40*0)-18=-18,-18,13,"[15, 6, 2, 14, 2, 16, 13, 3, 10, 17, 13, 3, 10]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 13, 3, 10, 1]"
1,6+4+31=41,41,9,"[8, 12, 6, 12, 5, 3, 17, 6, 3]","[0, 0, 0, 0, 0, 0, 6, 3, 1]"
2,4*11+13=57,57,10,"[6, 14, 3, 3, 12, 3, 5, 17, 7, 9]","[0, 0, 0, 0, 0, 0, 0, 7, 9, 1]"
3,44*39*26=44616,44616,14,"[6, 6, 14, 5, 11, 14, 4, 8, 17, 6, 6, 8, 3, 8]","[0, 0, 0, 0, 0, 0, 0, 0, 6, 6, 8, 3, 8, 1]"
4,(39*42)+27=1665,1665,15,"[15, 5, 11, 14, 6, 4, 16, 12, 4, 9, 17, 3, 8, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 8, 8, 7, 1]"


# Hyper Parameters

|Hyperparameter|Meaning|Value|
|-|-|-|
|`batch_size`|Number of data samples in a single batch|64|
|`epochs`|Total number of epochs to train|10|
|`embed_dim`|Dimension of the word embeddings|256|
|`hidden_dim`|Dimension of the hidden state in each timestep of the LSTM|256|
|`lr`|Learning Rate|0.001|
|`grad_clip`|To prevent gradient explosion in RNNs, restrict the gradient range|1|

In [ ]:
batch_size = 128
epochs = 14
embed_dim = 256
hidden_dim = 512
lr = 0.001
grad_clip = 1

# Data Batching
- Use `torch.utils.data.Dataset` to create a data generation tool called  `dataset`.
- The, use `torch.utils.data.DataLoader` to randomly sample from the `dataset` and group the samples into batches.

- Example: 1+2-3=0
    - Model input: 1 + 2 - 3 = 0
    - Model output: / / / / / 0 &lt;eos&gt;  (the '/' can be replaced with &lt;pad&gt;)
    - The key for the model's output is that the model does not need to predict the next character of the previous part. What matters is that once the model sees '=', it should start generating the answer, which is '0'. After generating the answer, it should also generate&lt;eos&gt;

In [ ]:
class Dataset(torch.utils.data.Dataset):
    def __init__(self, sequences):
        self.sequences = sequences

    def __len__(self):
        # return the amount of data
        return len(self.sequences)# Write your code here

    def __getitem__(self, index):
        # Extract the input data x and the ground truth y from the data
        x = self.sequences.iloc[index]['char_id_list']# Write your code here
        y = self.sequences.iloc[index]['label_id_list']# Write your code here
        return x, y

# collate function, used to build dataloader
def collate_fn(batch):
    batch_x = [torch.tensor(data[0]) for data in batch]
    batch_y = [torch.tensor(data[1]) for data in batch]
    batch_x_lens = torch.LongTensor([len(x) for x in batch_x])
    batch_y_lens = torch.LongTensor([len(y) for y in batch_y])

    # Pad the input sequence
    pad_batch_x = torch.nn.utils.rnn.pad_sequence(batch_x,
                                                  batch_first=True,
                                                  padding_value=char_to_id['<pad>'])

    pad_batch_y = torch.nn.utils.rnn.pad_sequence(batch_y,
                                                  batch_first=True,
                                                  padding_value=char_to_id['<pad>'])

    return pad_batch_x, pad_batch_y, batch_x_lens, batch_y_lens

In [ ]:
ds_train = Dataset(df_train[['char_id_list', 'label_id_list']])

In [ ]:
# Build dataloader of train set and eval set, collate_fn is the collate function
dl_train = torch.utils.data.DataLoader(ds_train, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)# Write your code here




# Model Design

## Execution Flow
1. Convert all characters in the sentence into embeddings.
2. Pass the embeddings through an LSTM sequentially.
3. The output of the LSTM is passed into another LSTM, and additional layers can be added.
4. The output from all time steps of the final LSTM is passed through a Fully Connected layer.
5. The character corresponding to the maximum value across all output dimensions is selected as the next character.

## Loss Function
Since this is a classification task, Cross Entropy is used as the loss function.

## Gradient Update
Adam algorithm is used for gradient updates.

In [ ]:
class CharRNN(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(CharRNN, self).__init__()
        # Write your code here
        # Design your own model, which should contain embedding layer, several hidden layers and a linear layer

        self.embedding = torch.nn.Embedding(num_embeddings=vocab_size,
                            embedding_dim=embed_dim,
                            padding_idx=char_to_id['<pad>'])

        self.lstm = torch.nn.LSTM(input_size=embed_dim,
                        hidden_size=hidden_dim,
                        num_layers=3,
                        dropout=0.1,
                        batch_first=True)

        self.linear = torch.nn.Sequential(torch.nn.Linear(in_features=hidden_dim,
                                    out_features=hidden_dim),
                                    torch.nn.ReLU(),
                                    torch.nn.Linear(in_features=hidden_dim,
                                    out_features=vocab_size))

    def forward(self, batch_x, batch_x_lens):
        return self.encoder(batch_x, batch_x_lens)

    # The forward pass of the model
    def encoder(self, batch_x, batch_x_lens):
        batch_x = self.embedding(batch_x)

        batch_x = torch.nn.utils.rnn.pack_padded_sequence(batch_x,
                                  batch_x_lens,
                                  batch_first=True,
                                  enforce_sorted=False)

        batch_x, _ = self.lstm(batch_x)

        batch_x, _ = torch.nn.utils.rnn.pad_packed_sequence(batch_x,batch_first=True)

        batch_x = self.linear(batch_x)

        return batch_x

    def generator(self, start_char, max_len=200):

        char_list = [char_to_id[c] for c in start_char]

        next_char = None

        while len(char_list) < max_len:
            # Write your code here
            # Pack the char_list to tensor
            # Input the tensor to the embedding layer, RNN layers, linear respectively
            input_tensor = torch.tensor([char_list], dtype=torch.long, device=next(self.parameters()).device)
            out = self.embedding(input_tensor)
            out, _ = self.lstm(out)
            out = self.linear(out)
            y = out[0, -1, :] # Obtain the next token prediction y

            next_char = torch.argmax(y).item() # Use argmax function to get the next token prediction

            if next_char == char_to_id['<eos>']:
                break

            char_list.append(next_char)

        return [id_to_char[ch_id] for ch_id in char_list]

In [ ]:
torch.manual_seed(2)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')# Write your code here. Specify a device (cuda or cpu)

model = CharRNN(vocab_size,
                embed_dim,
                hidden_dim)

In [ ]:
criterion = torch.nn.CrossEntropyLoss(ignore_index=char_to_id['<pad>'])# Write your code here. Cross-entropy loss function. The loss function should ignore <pad>
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)# Write your code here. Use Adam or AdamW for Optimizer


# Training
1. The outer `for` loop controls the `epoch`
    1. The inner `for` loop uses `data_loader` to retrieve batches.
        1. Pass the batch to the `model` for training.
        2. Compare the predicted results `batch_pred_y` with the true labels `batch_y` using Cross Entropy to calculate the loss `loss`
        3. Use `loss.backward` to automatically compute the gradients.
        4. Use `torch.nn.utils.clip_grad_value_` to limit the gradient values between `-grad_clip` &lt; and &lt; `grad_clip`.
        5. Use `optimizer.step()` to update the model (backpropagation).
2.  After every `1000` batches, output the current loss to monitor whether it is converging.

In [ ]:
from tqdm import tqdm
from copy import deepcopy
model = model.to(device)
model.train()
i = 0
for epoch in range(1, epochs+1):
    # The process bar
    bar = tqdm(dl_train, desc=f"Train epoch {epoch}")
    for batch_x, batch_y, batch_x_lens, batch_y_lens in bar:
        # Write your code here
        # Clear the gradient
        optimizer.zero_grad()

        batch_pred_y = model(batch_x.to(device), batch_x_lens)

        # Write your code here
        # Input the prediction and ground truths to loss function
        loss = criterion(batch_pred_y.view(-1, vocab_size), batch_y.to(device).view(-1))
        # Back propagation
        loss.backward()

        torch.nn.utils.clip_grad_value_(model.parameters(), grad_clip) # gradient clipping

        # Write your code here
        # Optimize parameters in the model
        optimizer.step()

        i+=1
        if i%50==0:
            bar.set_postfix(loss = loss.item())

    # Evaluate your model
    matched = 0
    total = 0
    bar_eval = tqdm(df_eval.iterrows(), desc=f"Validation epoch {epoch}")
    for _, row in bar_eval:
        batch_x = row['src']
        batch_y = row['tgt']

        prediction = model.generator(batch_x)# An example of using generator: model.generator('1+1=')

        # Write your code here. Input the batch_x to the model and generate the predictions
        # Write your code here.
        # Check whether the prediction match the ground truths
        # Compute exact match (EM) on the eval dataset
        # EM = correct/total
        pred_ans = ''.join(prediction[len(batch_x):])
        if pred_ans == str(batch_y):
            matched += 1
        total += 1


    print(matched/total)

Train epoch 1: 100%|██████████| 7813/7813 [02:18<00:00, 56.26it/s, loss=0.496]
Validation epoch 1: 10000it [00:49, 200.60it/s]


0.4494


Train epoch 2: 100%|██████████| 7813/7813 [02:16<00:00, 57.30it/s, loss=0.346]
Validation epoch 2: 10000it [00:49, 202.37it/s]


0.5615


Train epoch 3: 100%|██████████| 7813/7813 [02:17<00:00, 56.95it/s, loss=0.31]
Validation epoch 3: 10000it [00:50, 199.36it/s]


0.6021


... (18 outputs left out)

Train epoch 13: 100%|██████████| 7813/7813 [02:17<00:00, 56.97it/s, loss=0.111]
Validation epoch 13: 10000it [00:49, 201.34it/s]


0.8753


Train epoch 14: 100%|██████████| 7813/7813 [02:17<00:00, 56.86it/s, loss=0.118]
Validation epoch 14: 10000it [00:49, 201.26it/s]

0.8701


Running environment: colab

Python version: colab

##### 1. Describe your model architecture, data preprocessing methods, and hyperparameter settings, and justify your choices.  (5%)

Answer:

1. 字元嵌入層(Embedding Layer)=256 因為字典總共包含18個字元數量非常少，256維足夠讓模型區分數字與運算符號的語意
2. 循環神經網路使用3層的LSTM`（hidden_dim = 512, num_layers = 3, dropout = 0.1）`：
    * (1) 選LSTM因為算術題目常常包含括號和跨位數進位，序列較長。RNN容易在傳遞過程中發生梯度消失，而LSTM的Cell State和門控機制能在長距離運算表現更好
    * (2) 3層是因為加減乘除有先後運算順序，多層結構能提供不同層度的特徵推演
    * (3) Hidden 512是因為我認為要記住中間的計算結果和進位，所以增加維度記住更多特徵
    * (4) dropout = 0.1在LSTM層與層之間隨機丟棄10%，避免神經元之間互相依賴，防止模型死記硬背訓練資料，提升在測試集上的泛化能力
3. 線性分類層：
    * Linear(512, 512) ➔ ReLU() ➔ Linear(512, 18)。
    * 預測字元前多加一層非線性映射，幫助模型將512維的高維隱藏狀態更好地解耦，轉換成18個字元的機率分佈。
4. 超參數:
    * batch_size = 128：
    因我使用colab較好的GPU為了加快訓練速度讓Batch涵蓋更題型，梯度更新更加平穩
    * epochs = 14：
    模型維度加大到512維與3層後，需要足夠的回合訓練。實測中發現模型在8輪內進步最快，第10到14輪在86%~87%收斂
    * lr = 0.001：
    採用最經典參數
    * grad_clip = 1.0：
    RNN在長序列反向傳播時容易產生梯度爆炸。將梯度值限制在[-1.0, 1.0]之間避免梯度爆炸

<br>

##### 2. If you use 2-layer LSTM or GRU instead of RNN, what will happen to the quality of your answer generation? Why? (10%)

Answer:
在這次作業中，我實際在相同的超參數設定下（embed_dim=256, hidden_dim=256, batch_size=64, lr=0.001），分別訓練並測試了2-layer RNN與2-layer LSTM
<br>
以下是我在驗證集（10,000 筆資料）上跑出的評估結果：

<br>

| 模型架構 | 訓練輪次Epochs | 訓練損失Loss | 驗證集EM準確率 | 說明 |
| :--- | :---: | :---: | :---: | :--- |
| 2-layer RNN | 2 | 0.878 | 15.02% | 訓練兩輪後準確率即停滯在15%左右 |
| 2-layer LSTM | 2 | 0.330 | 57.32% | 同樣只跑兩輪，準確率直接增至57.32% |
| 2-layer LSTM | 10 | 0.181 | 72.50% | 隨著訓練輪數增加，準確率一路上升 |

<br>

從實驗輸出和案例中，RNN幾乎只能算對最簡單的短算式，一旦題目出現括號或多位數乘法，生成的答案經常錯誤。改用LSTM後，模型藉由記憶，從第一位數字到最後的 `<eos>` 結束符號能保持一致性與正確性。複雜題目的答案全對率大幅提升。


1. 反向傳播梯度流的本質差異：RNN隱藏狀態的梯度需要反覆連乘同一個權重矩陣W_hh 與tanh 的導數。當算式序列長度比較長時，梯度會衰減，導致無法學到長距離語法規則。相比之下，LSTM 引入Cell State在時間步之間的傳遞是加法而非矩陣連乘。使得誤差梯度在反向幾乎沒有衰減阻礙，能夠把長距離的算術邏輯完整回傳更新。
2. RNN只有單一的隱藏狀態h_t，每一個新的字元進來都會覆蓋掉原本的記憶，導致看見後面忘了前面。而LSTM有遺忘門可以選擇性遺忘題目中已經算完的中間過渡值。輸入門把關鍵的運算子寫入長期記憶。而輸出門在自回歸生成答案時，控制目前應該吐出哪個字元。

<br>

##### 3. If we construct an training set using three-digit numbers while the evaluation set is constructed from two-digit numbers, what will happen to the quality of your answer generation? (10%)

Answer:

python生成三萬筆三位數資料，用 `random.randint(100, 999)` 隨機產生三個介於100∼999之間的正整數。從 `['+', '-', '*']` 中用 `random.choice()`隨機挑兩個符號，跟提供的資料一樣沒有使用除法。且隨機套用 3 種括號模板`（pattern）`第一種是無括其他下兩個是括號在左跟括號在右，最後使用`eval()`算出答案。
我用這三萬筆三位數資料訓練相同的3-layer LSTM模型5個Epochs（Training Loss 由 `1.81` 下降至 `1.53`），接著用提供的一萬筆兩位數評估集（`arithmetic_eval.csv`，數值範圍0到49）進行測試。
結果發現，模型在兩位數評估集上的(EM)準確率直接掉到0.00%
一開始看到0.00%時我以為程式寫錯了，但把模型實際預測的結果印出來與真實答案對照後（如下表），我發現模型其實不是在亂猜，而是出現了很明顯的規律：

<br>

| 兩位數算式 | 真實答案 | 真實位數 | 三位數模型預測 | 預測位數 | 總結 |
| :--- | :--- | :---: | :--- | :---: | :--- |
| 34+47-32= | 49 | 2位數 | 571 | 3位數 | 答案被放大了約10倍，變成3位數 |
| 33-41-15= | -23 | 負2位數 | 147 | 3位數 | 連負號都沒預測出來，一樣輸出3位數 |
| 31-(36*44)= | -1553 | 負4位數 | -120077 | 負6位數 | 有成功判斷出負號，但數字變成6位數 |
| 21*(41+3)= | 924 | 3位數 | 240006 | 6位數 | 看到乘法就自動輸出 6 位數的答案 |
| (47-27)+26= | 46 | 2位數 | 577 | 3位數 | 答案也是三位數 |


<br>

1.  模型死記了答案的輸出長度<eos>偏移，在100到999的三位數訓練集中，加減法的答案幾乎都是3到4位數，而只要有乘法的算式，答案幾乎都是6位數。因此模型在生成答案時，已經習慣看到加減法要輸出至少3個數字才接`<eos>`、看到乘法要輸出6個數字才接<eos>。當測試題目變成兩位數時，模型不知道需要提早停下來，導致所有預測答案都比正確大，在嚴格比對字串的下自然全部算錯。
2.  其中最有趣的是第三題 `31-(36*44)=`（真實答案-1553），模型預測出-120077。這代表模型其實看得懂括號內先相乘後，前面的數字不夠減，答案會是負數，因此第一個字元就正確輸出了負號-；只是因為它把 `36*44` 當成三位數乘三位數，才吐出了負6位數。
3.  因為根據上述觀察，模型習慣每個數字固定是3個字元，因此我透過數字前面補0（例如把 `34+47-32=` 改成 `034+047-032=`），湊滿三個字元，看模型能不能算對？對照結果如下：

<br>

| 原始兩位數算式 | 補零後格式 | 真實答案 | 原始預測（無補零） | 補零後預測 | 總結 |
| :--- | :--- | :--- | :--- | :--- | :--- |
| 34+47-32= | 034+047-032= | 49 | 571 | 271 | 預測數值從500多降到200 多，但還是3位數 |
| 33-41-15= | 033-041-015= | -23 | 147 | -471 | 原本沒算出的負號 `-` 成功出現了 |
| 31-(36*44)= | 031-(036*044)= | -1553 | -120077 | -97004 | 維持負號，數值從12萬降到9.7萬 |
| 21*(41+3)=  | 021*(041+003)= | 924 | 240006 | 110046 | 數值從24萬直接降到11萬 |
| (47-27)+26= | (047-027)+026= | 46 | 577 | 301 | 數值從577降到301，但依然維持3位數 |

<br>

從結果可以看出，單純在測試時補0還是沒辦法讓模型直接算對兩位數。我認為原因是訓練資料是用`random.randint(100, 999)`產生的，模型在訓練階段根本沒有看過以0開頭的假三位數（`0xx`），而且訓練集的答案裡也幾乎沒有出現過兩位數。所以當模型在百位數看到沒見過的'0'時，只能把它當成訓練集中見過最小的三位數（約100到200的數值來算，這也解釋了為什麼補0之後，所有預測出來的數字都剛好縮小了一半左右（例如571變271、240006變110046）。
不過補0還是驗證了一件事：在 `33-41-15=` 這題中，原本因為兩位數長度太短導致模型連正負號都算錯（原本猜正數147）；但改成 `033-041-015=` 補齊三個字元寬度後，減號到了模型熟悉的位置，模型就立刻發現這題不夠減而成功吐出負號（`-471`）。因此要讓模型具備跨位數的泛化能力，必須在訓練階段就將資料範圍設為混合位數（如 `0到999`），或是從訓練時就統一把所有數字補0對齊，模型才能學會不同位數的運算與 `<eos>` 停止時機。


<br>

##### 4. If we construct a training set that includes 20% incorrect answers, how will this affect the quality of the generated responses? Present some examples. (10%)

Answer:

 我是直接使用提供的一百萬筆訓練集（`arithmetic_train.csv`），使用 `np.random.choice()` 隨機抽出其中20%（共200000筆）的資料，並用 `random.randint(-500, 500)` 來替換掉原本的 `tgt`，剩下80%（800,000 筆）則維持不變。
我使用與主實驗完全相同的3-layerLSTM模型（`hidden_dim=512`）訓練5個Epochs，並在評估集（`arithmetic_eval.csv`）上與原本100%正確訓練集跑出來的結果進行對照：

<br>

| Epoch | 乾淨訓練集Loss | 20%錯誤訓練集Loss | 乾淨訓練集Eval EM | 20%錯誤訓練集Eval EM | 差異 |
| :---: | :---: | :---: | :---: | :---: | :---: |
| **Epoch 1** | 0.496 | 0.812 | 44.94% (0.4494) | 45.96% (0.4596) | +1.02% |
| **Epoch 2** | 0.346 | 0.720 | 56.15% (0.5615) | 55.42% (0.5542) | -0.73%
| **Epoch 3** | 0.310 | 0.776 | 60.21% (0.6021) | 61.69% (0.6169) | +1.48% |
| **Epoch 4** | 0.251 | 0.706 | 69.65% (0.6965) | 66.63% (0.6663) | -3.02% |
| **Epoch 5** | 0.248 | 0.688 | **71.10% (0.7110)** | **68.28% (0.6828)** | **-2.82%** |

<br>
---
<br>



| 預測結果分類 | 測試算式src | 真實答案tgt | 20%雜訊模型預測pred | 總結 |
| :---: | :--- | :---: | :---: | :--- |
| 正確 | 34+47-32= | 49 | 49 | 純加減法、答案為兩位數，完全算對 |
| 正確 | 33-41-15= | -23 | -23 | 負數的純減法，完全算對 |
| 正確 | (47-27)+26= | 46 | 46 | 含括號的加減法，完全算對 |
| 正確 | 21*(41+3)= | 924 | 924 | 乘數較小三位數以內的乘法，正確率會比較好 |
| 錯誤 | 19*35+33= | 698 | 692 | 百差個位數（差6） |
| 錯誤 | 31-(36*44)= | -1553 | -1555 | 負號與前三位-155全對，只差個位數 |
| 錯誤 | (46*38)+25= | 1773 | 1797 | 差後面兩位數24 |
| 測錯誤 | 5*29*19= | 2755 | 2835 | 連乘算式三位進位算錯 |
| 錯誤 | 47*18*34= | 28764 | 29722 | 連乘算式答案高達五位數，僅萬2字正確 |
---
#### 分析:
1. 整體準確率依然維持不錯（68.28%），原本我以為混入20%的錯誤答案會讓模型崩潰，但結果顯示模型在Epoch 5還是達到了`68.28%`的準確率（比乾淨訓練集的`71.10%`下降約 `2.82%`）。80%的正確資料依然提供一致且正確的算術規則，而20%的錯誤答案是隨機分散在 `[-500, 500]` 之間沒有規律。因此在推論時透過 `argmax` 挑選機率最高的字元，80%的正確訊號還是能壓過分散的隨機雜訊。不過在訓練過程中，這20%無法被預測的隨機標籤會產生很大的梯度雜訊，導致 Training Loss 從第 2 個 Epoch 開始就卡在 `0.68 ~ 0.77` 左右（乾淨資料集可以順利降到 `0.248`），且到了後期第四以及第五輪準確率的成長開始落後於乾淨模型。
2. 表格可以發現一個很明顯的現象，加減法幾乎都能全對，但出錯的題目都包含乘法，而且**真實答案的數值越大，模型就越難算對**。我認為原因在於加減法的答案範圍相對較小，就算有20%錯誤答案也不影響模型學習。相反大數字乘法如四位數、五位數答案的數值組合非常多，每一種進位組合在訓練集裡出現的次數本來就比較少，當再被抽走20%換成錯誤答案時，模型就很難把複雜的多位數乘法學得很精準。
3. 另一個很有趣的發現是，就算模型算錯了，它的預測值跟真實答案其實非常接近，大多只差在最後一兩位數字。因為模型是由左往右生成答案，要估計出答案的正負號、位數長度和最高位數相對容易，即使20%雜訊也能學好；但要算對最末端的十位數與個位數，模型必須在隱藏狀態中完成精確的進位計算，而20%的標籤雜訊剛好破壞了模型在 **"後期"** 收斂的能力，才出現只有尾巴1～2位數算錯的現象。

<br>

##### 5. Why do we need gradient clipping during training? (5%)

Answer:
為了解決隨著時間反向傳播BPTT所導致的梯度爆炸
1. 主要是RNN在處理時間序列資料時，每個時間步重複使用的是同一套記憶權重矩陣$W$。例如本次實作作業的算術題目長度約為15到35個字元。當計算Loss鏈鎖律往回傳誤差時，梯度需不斷乘上同一套權重，形成如 $(W^T)^{T-1}$ 的高次方連乘。且在256維的高維矩陣中，為了記住前面算式避免梯度消失，必然會讓矩陣的最大特徵值大於1，導致誤差回傳後，梯度會呈指數級暴增。因此若沒有限制，梯度在乘上學習率（$\Delta W = -lr \times g$）後，會讓權重震盪破壞模型先前已學到的記憶，更導致浮點數計算溢位，使訓練損失直接變成 `Loss = NaN` 或 `Inf`，導致訓練徹底中斷。
2. 在實作中採用了 `torch.nn.utils.clip_grad_value_(model.parameters(), grad_clip=1)`對模型中所有參數的梯度純量進行截斷，強制限制在 $[-1, 1]$ 區間內（若 $g > 1 \to 1$；若 $g < -1 \to -1$；介於之間則保持原樣）。
保證了每個參數在單一 Batch中的最大變動量最多不超過 $ lr \times 1 = 0.001$，徹底避免劇烈動盪。另外若使用另一種按向量總長度縮放`clip_grad_norm_`可以保持方向不變，相比 `clip_grad_value_` 因為是每個分量各自截斷可能會稍微微調梯度的方向角度。


##### … Anything that can strengthen your report. (5%)

Answer:
<br>
為了進一步探討模型的極限針對主模型（3-layer LSTM, hidden_dim=512, batch_size=128）進行了14輪訓練，並分析了資料集中的分佈情況，進而提出具體的改進與資料增強方向：

1. 14輪訓練收斂

    下表為模型在驗證集上的歷程：

    | 訓練輪次Epoch | 結尾Training Loss | 驗證集EM準確率 |
    | :--- | :---: | :---: |
    | Epoch 1 | 0.496 | 44.94% |
    | Epoch 2 | 0.346 | 56.15% |
    | Epoch 4 | 0.251 | 69.65% |
    | Epoch 6 | 0.283 | 72.13% |
    | Epoch 8 | 0.193 | 77.46% |
    | Epoch 10 | 0.084 | 82.97% |
    | Epoch 12 | 0.103 | 86.67% |
    | Epoch 13 | 0.111 | 87.53% |
    | Epoch 14 | 0.118 | 87.01% |

    模型在第13輪達到87.53%的最佳成績。到了第14輪時降下0.52%，說明模型在現有資料下已達收斂極限。在第12到13輪停止可以取得最佳泛化能力並節省訓練資源。
<br>

2. 數據分佈加強

    為了探究剩下的12.47%錯誤成因，我寫了Python腳本對訓練集arithmetic_train.csv進行了分佈統計，發現資料在連續加法/減法/乘法較少：


    | 運算子 | 訓練集出現次數 | 佔比 (%) | 括號分佈情況 |
    | :--- | :--- | :--- | :--- |
    | + - / - + | 各約 15.8 萬筆 | ~15.8% | 左/右/無括號各佔 33% |
    | * - / - * | 各約 13.7 萬筆 | ~13.7% | 左/右/無括號各佔 33% |
    | + * / * + | 各約 13.3 萬筆 | ~13.3% | 左/右/無括號各佔 33% |
    | - -（連續減法） | 僅 53,042 筆 | 5.30% | 無括號佔 100%（括號 0%） |
    | + +（連續加法） | 48,929 筆 | 4.89% | 無括號佔 100% 樣本較少 |
    | * *（連續乘法） | 37,102 筆 | 3.71% | 無括號佔 100% 樣本最少 |


    連續減法的樣本僅佔5.3%，模型接觸到的經驗大幅少於混合運算。
    加減乘中唯獨減法不具備結合律，在沒有括號下，模型在看到等號的瞬間必須精確判斷兩次相減是否跌破0、該不該吐出負號-，造成正負號決策困難。
<br>

後來自行撰寫腳本合成了20萬筆擴充資料（補強連續減法10萬筆、加法5萬筆、乘法5萬筆，並特別加入a-(b-c)=等括號變形），將訓練集擴展至120萬筆重新訓練14輪，我的發現是可能題目過於複雜相比原本的訓練資料模型需要更長的訓練輪數，且在訓練過程中明顯收斂速度慢了很多，以至於相同14輪下最後在驗證集是83.61%，並沒有超越基準模型。因此反思了神經網路學習數學的根本瓶頸，神經網路本質是模糊連續的機率估計器，用浮點數矩陣硬擬合精確運算似乎過於為難了。


##### Generative AI Usage Statement

Answer:
* 使用工具：Gemini
* 使用範圍：
    * 概念與數學理解：請教 NN在隨時間反向傳播（BPTT）時權重共享與連乘導致梯度爆炸的原理，以及clip_grad_value_梯度裁剪的幾何意涵。
    * PyTorch語法請教：釐清Dataset、DataLoader的collate_fn打包機制，以及自回歸 generator()中切片語法與torch.argmax的運作邏輯。
    * 資料分析：使用Python腳本統計arithmetic_train.csv的運算子分佈